In [ ]:
import os
from google.colab import userdata

if "HF_TOKEN" in os.environ or userdata.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset =load_dataset("CShorten/ML-ArXiv-Papers",split='train')

In [ ]:
print(dataset)

In [ ]:
dataset[0]

In [ ]:
import pandas as pd

### Re-initializing DataFrame `df`

It appears the `df` DataFrame was lost, likely due to a runtime restart. The following cells will re-load the dataset and reconstruct `df` with the necessary columns and cleaning steps.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
df = pd.DataFrame(dataset)
df = df[['title', 'abstract']]
df = df.head(15000).copy()
df["paper_text"] = df["title"] + " " + df["abstract"]
df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

The `df` DataFrame is now re-initialized and ready for use.

In [ ]:
df=pd.DataFrame(dataset)
df

In [ ]:
df.columns

In [ ]:
df=df[['title','abstract']]

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df=df.head(15000).copy()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df["paper_text"]= df["title"]+ " "+df["abstract"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
type(df[["paper_text"]])

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
# To fix persistent import errors due to version conflicts, perform a clean reinstallation.
# It's CRITICAL to restart the runtime after running this cell.
!pip uninstall -y transformers tokenizers peft sentence-transformers
!pip install --force-reinstall transformers==4.41.0
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
print(type(model))

In [ ]:
try:
    _ = df.iloc[0]
except NameError:
    print("DataFrame 'df' not found. Re-initializing from dataset...")
    import pandas as pd
    from datasets import load_dataset

    # Load dataset
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')

    # Create DataFrame and apply necessary cleaning/selection steps
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']]
    df = df.head(15000).copy()
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("DataFrame 'df' re-initialized successfully.")

sample_text=df["paper_text"].iloc[0]
sample_text

In [ ]:
df["paper_text"]=df ["paper_text"].str.replace("\n","", regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [ ]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)


In [ ]:
embedding[:56]

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)

In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)

In [ ]:
embedding = model.encode(df["paper_text"].to_list(), batch_size=32, show_progress_bar=True)

In [ ]:
print(embedding.shape)
print(type(embedding))

In [ ]:
embedding.dtype

In [ ]:
!pip install faiss-cpu
import faiss

In [ ]:
faiss.normalize_L2(embedding)

In [ ]:
index=faiss.IndexFlatIP(384)

In [ ]:
index.add(embedding)

In [ ]:
print(index.ntotal)

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode(query)
query_embedding.reshape(1,-1)
query_embedding.shape

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

In [ ]:
if query_embedding.ndim == 1:
    query_embedding = query_embedding.reshape(1, -1)
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

In [ ]:
print(df.iloc[10466]["title"])

In [ ]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
  print("Loading saved embeddings")
  embeddings = np.load("paper_embeddings.npy")
else:
  print("Generating embeddings")
  embeddings = model.encode(df["paper_text"].tolist(), batch_size=32, show_progress_bar=True)
  np.save("paper_embeddings.npy", embeddings)
  print("Embeddings saved successfully!")#embeddings = model.encode(
      #df["paper_text"].tolist(),
      #batch_size=32,
      #show_progress_bar=True
    #)

In [ ]:
print(embedding.shape)
print(type(embeddings))

In [ ]:
import faiss

In [ ]:
#faiss.normalize_L2(embedding)

In [ ]:
#index=faiss.IndexFlatIP(384)

In [ ]:
#index.add(embedding)

In [ ]:
if os.path.exists("paper_faiss.index"):
  print("Loading existing FAISS index")
  index = faiss.read_index("paper_faiss.index")
else:
  print("Creating new FAISS index")
  faiss.normalize_L2(embeddings)
  index = faiss.IndexFlatIP(384)
  index.add(embeddings)
  faiss.write_index(index,"paper_faiss.index")
  print("FAISS index saved successfully!")

In [ ]:
# [Full messages cannot be displayed on this version]

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
print(df.iloc[11873]["title"])

In [ ]:
import faiss
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  return D,I

In [ ]:
D,I= search_paper("deep learning for medical image analysis")
print(D)
print(I)

In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  return D,I

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import os

# Re-initialize the model to ensure it's available
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Re-load the FAISS index to ensure it's available
if os.path.exists("paper_faiss.index"):
  index = faiss.read_index("paper_faiss.index")
else:
  print("Warning: FAISS index file 'paper_faiss.index' not found. Please ensure it has been created and saved, or re-run the index creation cells.")
  # Depending on the criticality, you might want to raise an error here.
  # For this fix, we assume the file will be present if previous steps ran.

# Redefine the search_paper function
def search_paper(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D, I = index.search(query_embedding, k)
  return D, I

# Call the search_paper function
D, I = search_paper("deep learning for medical image analysis")
print(D)
print(I)

In [ ]:
!pip install transformers==4.46.3

In [ ]:
!pip uninstall -y transformers tokenizers peft sentence-transformers
!pip install --force-reinstall transformers==4.41.0
!pip install sentence-transformers
!pip install peft # Reinstall peft if it was a direct dependency or used elsewhere
from transformers import pipeline

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
type(summarizer)

In [ ]:
# Ensure df is defined. If not, re-create it from the dataset.
# This block re-creates df if it's not present, accounting for potential kernel restarts.
try:
    _ = df.iloc[0]
except NameError:
    print("DataFrame 'df' not found. Re-initializing from dataset...")
    import pandas as pd
    from datasets import load_dataset

    # Load dataset (assuming 'CShorten/ML-ArXiv-Papers' is available)
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')

    # Create DataFrame and apply necessary cleaning/selection steps as per previous cells
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']]
    df = df.head(15000).copy() # Use .copy() to avoid SettingWithCopyWarning
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("DataFrame 'df' re-initialized successfully.")

# Now df is guaranteed to exist, proceed with summarization
summary = summarizer(df.iloc[10466]["abstract"])
print(summary)

In [ ]:
summary=summarizer (df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

In [ ]:
type(summary)

In [ ]:
type(summary[0])

In [ ]:
summary[0]["summary_text"]

In [ ]:
# To fix persistent import errors due to version conflicts, perform a clean reinstallation.
# It's highly recommended to restart the runtime after running this cell once.
!pip uninstall -y transformers tokenizers peft sentence-transformers
!pip install --force-reinstall transformers==4.46.3
!pip install sentence-transformers
!pip install peft # Reinstall peft if it was a direct dependency or used elsewhere

from sentence_transformers import SentenceTransformer
import faiss # Ensure faiss is imported here if not already in search_paper definition
import os # Needed for os.path.exists

# Re-initialize the model to ensure it's available
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Re-initialize the summarizer pipeline, as it also depends on transformers
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

# Load the FAISS index to ensure 'index' is defined
if os.path.exists("paper_faiss.index"):
  index = faiss.read_index("paper_faiss.index")
else:
  # This case should ideally not happen if the index was saved correctly earlier
  # but including for robustness. You might need to re-run index creation cells.
  print("Error: FAISS index file not found. Please re-create the index.")
  # You might want to exit or raise an error here if the index is critical

D, I = search_paper("deep learning for medical image analysis")

for score, idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx] ["abstract"][:500])

    summary=summarizer(df.iloc[idx] ["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[0]["summary_text"])
    print()

In [ ]:
def search_and_summarize(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score",score)
    print("Title",df.iloc[idx]["title"])

    abstract_text = df.iloc[idx]["abstract"]
    if pd.isna(abstract_text) or not abstract_text.strip():
      print("Abstract: [Abstract not available]")
      print()
    else:
      print("Abstract",abstract_text[:500])
      print()
      summary=summarizer(abstract_text,max_length=120,min_length=40,do_sample=False)
      print(summary)
      print(summary[0]["summary_text"])
      print()

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import os
import pandas as pd
from datasets import load_dataset
from transformers import pipeline

# Ensure df is defined. If not, re-create it from the dataset.
try:
    _ = df.iloc[0]
except (NameError, AttributeError):
    print("DataFrame 'df' not found or invalid. Re-initializing from dataset...")
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']]
    df = df.head(15000).copy()
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("DataFrame 'df' re-initialized successfully.")

# Re-initialize the model to ensure it's available
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Re-initialize the summarizer pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

# Re-load the FAISS index to ensure it's available
# Assuming 'paper_faiss.index' exists from previous successful runs
if os.path.exists("paper_faiss.index"):
  index = faiss.read_index("paper_faiss.index")
else:
  print("Error: FAISS index file 'paper_faiss.index' not found. Please ensure it has been created and saved, or re-run the index creation cells.")
  # This is a critical error, you might want to stop execution or handle it more robustly

# The search_and_summarize function must be defined before calling it.
# Assuming it's defined in a preceding cell (Pk4JVoylCUSD), if not, its definition should be included here.
# For now, we assume it's globally available from Pk4JVoylCUSD which was executed.

search_and_summarize("Deep learning in medical image analysis",k=5)

In [ ]:
!pip install keybert
from keybert import KeyBERT

In [ ]:
kw_model = KeyBERT()

In [ ]:
type(kw_model)

In [ ]:
type(kw_model)#model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
pip install keybert==0.8.5

In [ ]:
try:
    _ = df.iloc[0]
except NameError:
    print("DataFrame 'df' not found. Re-initializing from dataset...")
    import pandas as pd
    from datasets import load_dataset

    # Load dataset
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')

    # Create DataFrame and apply necessary cleaning/selection steps
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']]
    df = df.head(15000).copy() # Use .copy() to avoid SettingWithCopyWarning
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("DataFrame 'df' re-initialized successfully.")

print(df.iloc[10466]["abstract"])

In [ ]:
if 'df' not in locals():
    print("DataFrame 'df' not found. Re-initializing from dataset...")
    import pandas as pd
    from datasets import load_dataset
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']]
    df = df.head(15000).copy()
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("DataFrame 'df' re-initialized successfully.")

if 'kw_model' not in locals():
    from keybert import KeyBERT
    kw_model = KeyBERT()
    print("KeyBERT model re-initialized successfully.")

document = df.iloc[0]['paper_text'] # Use the first paper's text as a sample document
keywords = kw_model.extract_keywords(document)
print(keywords)

In [ ]:
print(keywords)

In [ ]:
print(type(keywords))

In [ ]:
print(type(keywords[0]))

In [ ]:
kw_model.extract_keywords(document, keyphrase_ngram_range=(1,3))

In [ ]:
print(keywords)

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
from datasets import load_dataset

# Ensure df is defined. If not, re-create it from the dataset.
try:
    _ = df.iloc[0]
except (NameError, AttributeError):
    print("DataFrame 'df' not found or invalid. Re-initializing from dataset...")
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')
    df = pd.DataFrame(dataset)
    df = df[['title', 'abstract']]
    df = df.head(15000).copy()
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", "", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    print("DataFrame 'df' re-initialized successfully.")

# 1. Model ko load karo
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Sirf 2000 papers ka data embedding banao taaki 1 minute me chal jaye
df_small = df.head(2000).copy()

print("Embedding generation shuru ho rahi hai, bas 1-2 minute rukye...")
embeddings = model.encode(df_small["paper_text"].to_list(), batch_size=64, show_progress_bar=True)
print("Safar safal raha! Embeddings successfully ban gayi hain. ")

In [ ]:
from keybert import KeyBERT
from transformers import pipeline
import pandas as pd
import numpy as np

In [ ]:
kw_model = KeyBERT(model='all-MiniLM-L6-v2')

In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
def research_trend_analyzer(query, top_k=5):

    query_embedding = model.encode([query])

    D, I = index.search(query_embedding.astype("float32"), top_k)

    papers = df.iloc[I[0]]

    combined_text = " ".join(papers["abstract"].tolist())

    keywords = kw_model.extract_keywords(
        combined_text,
        top_n=10,
        stop_words='english'
    )

    summary = summarizer(
        combined_text[:2000],
        max_length=120,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]

    print("="*60)
    print("Research Topic :", query)

    print("\nTop Keywords:")
    for k in keywords:
        print("-", k[0])

    print("\nResearch Summary:")
    print(summary)

    print("\nFuture Scope:")
    print("- Multimodal AI")
    print("- Efficient Transformer Models")
    print("- Domain-specific LLMs")
    print("- Explainable AI")
    print("- Green AI")

In [ ]:
research_trend_analyzer("Large Language Models")

In [ ]:
research_trend_analyzer("Machine Learning")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compare_papers(title1, title2):

    paper1 = df[df["title"] == title1]
    paper2 = df[df["title"] == title2]

    if len(paper1)==0 or len(paper2)==0:
        print("Paper not found.")
        return

    idx1 = paper1.index[0]
    idx2 = paper2.index[0]

    emb1 = embeddings[idx1].reshape(1,-1)
    emb2 = embeddings[idx2].reshape(1,-1)

    similarity = cosine_similarity(emb1, emb2)[0][0]

    print("="*70)

    print("📄 Paper 1")
    print(paper1["title"].values[0])

    print("\n📄 Paper 2")
    print(paper2["title"].values[0])

    print("\n Similarity Score : {:.2f}%".format(similarity*100))

    print("\nAbstract Comparison")

    print("\nPaper 1:")
    print(paper1["abstract"].values[0][:500],"...")

    print("\nPaper 2:")
    print(paper2["abstract"].values[0][:500],"...")

    print("\nRecommendation")

    if similarity>0.80:
        print("These papers are highly related and discuss similar research directions.")

    elif similarity>0.60:
        print("These papers are moderately related with overlapping concepts.")

    else:
        print("These papers focus on different research problems.")

In [ ]:
df["title"].head(10)

In [ ]:
compare_papers(
    df["title"].iloc[0],
    df["title"].iloc[1]
)

In [ ]:
print("\nWhy this paper is recommended?")

print("• High semantic similarity with your query.")
print("• Similar research concepts.")
print("• Closely related methodology.")
print("• Relevant keywords matched.")

In [ ]:
print("\n Why this paper is recommended?")

print("• High semantic similarity with your search query.")
print("• Similar research concepts and keywords.")
print("• Closely related methodology.")
print("• Relevant to the user's research topic.")

In [ ]:
print("="*50)
print(" Research Collection Statistics")
print("="*50)

print("Total Papers :", len(df))
print("Embedding Dimension :", embeddings.shape[1])
print("Vector Database : FAISS")
print("Embedding Model : all-MiniLM-L6-v2")
print("Summarization Model : facebook/bart-large-cnn")
print("Keyword Extraction : KeyBERT")